## Term Search Method

> Introduction: we are looking to perform NLP to phenotype the following symptoms > between Jan 1, 2010 and Dec 31, 2024 among eligible patients (i.e., patients with > hypertension). Eligible patients will be identified from the VUMC site of STAR CRN > and MRN of eligible patients will be provided by STAR CDRN analyst to link and > identify eligible patients. 
> 
> Note: Report results in 7 columns as “patient_ID”, “keyword”, “date of encounter > key word identified”, “EHR document type section the key word was identified (if > they have document type)”, “a sentence where key word identified (e.g., 20 words > before and 20 words after key word)”, and a column  for “negation related > indicator”  
> 
> Feel free to suggest any improvements you believe could enhance the extraction > process.
>  
> Synonyms for key words are provided in the first column. You can use more synonyms > based on your experience

## Sources

| **Inclusion Criteria**                           | **Defined in View/Table**                   | **Materialized Table for Testing**                       |
|--------------------------------------------------|---------------------------------------------|----------------------------------------------------------|
| eligible patients                                | cohort                              |                                                          |
| symptoms > between Jan 1, 2010 and Dec 31, 2024  | vw_notes_with_timeframe_for_cohort   | materialized_vw_notes_with_timeframe_for_cohort  |


## Features

| **Feature** | **Variable** |
|-------------|--------------|
| “patient_ID” | patient_id |
| “keyword” | search_term |
| “date of encounter > key word identified” | encounter_date_gt_key_word_identified_date |
| “EHR document type section the key word was identified (if > they have document type)” | note_type |
| “a sentence where key word identified (e.g., 20 words > before and 20 words after key word)” | text_window |
| “negation related > indicator” | polarity |

## Definitions

victr_rd.rd_omop_prod - VUMC Research Derivative (RD), which leverages the the [OMOP CDM](https://www.ohdsi.org/data-standardization/).
note groups - a class of notes (e.g., ECG Impression, Outpatient Notes, etc.) defined by characteristics specific to each type (e.g., note title, note concept type, etc.).

## Algorithm

- Filter notes to those found in at least one of the note groups defined in `default.vw_note_id_to_note_groups` (`default.materialized_vw_note_id_to_note_groups` for testing).


In [0]:
%pip install --upgrade medspacy
%pip install --upgrade loguru
%pip install --upgrade threadpoolctl
%pip install nlp_preprocessor
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [0]:
# %pip show medspacy
# %pip show loguru
# %pip show threadpoolctl
# %pip show nlp_preprocessor

In [0]:
dbutils.library.restartPython()

In [0]:
import spacy
from spacy.tokens import Doc
from spacy.tokenizer import Tokenizer
from medspacy.ner import TargetRule
from pyspark.sql import DataFrame as pysqlDataFrame
from typing import Any, Generator, Iterator, List
from dataclasses import dataclass
import pandas as pd
from typing import Callable


# logger.remove()

TESTING = True
WINDOW_SIZE = 20
TARGET_ENTITY_LABEL = "SYMPTOM"

if TESTING:
    TEST_DATA_FOR_VALIDATION = None

from IPython.display import display, HTML


def testing_log(section: str, obj_display=None):
    if TESTING:
        section_suffix = ""

        display_html = ""
        if obj_display is not None:
            if isinstance(obj_display, (List, pd.DataFrame)):
                section_suffix = " (first 3)"
                if isinstance(obj_display, List):
                    display_html = f"""
                    <ul>
                    {''.join(f'<li>"{x}"</li>' for x in obj_display[0:3])}
                    </ul>
                    """
                elif isinstance(obj_display, pd.DataFrame):
                    display_html = obj_display.head(3).to_html()
            else:
                display_html = str(obj_display)

        top_line_style = "margin-bottom: 0px"
        bottom_line_style = "margin-top: 0px; " + (
            "margin-bottom: 0px" if display_html else ""
        )
        display_html = f"""
        <div style='{top_line_style}'>-------------------------------------------------------------------</div>
        <h4 style="margin-top: 0px; margin-bottom: 0px;">{section}{section_suffix}</h4>
        <div style='{bottom_line_style}'>-------------------------------------------------------------------</div>
        {display_html}
        """

        display(HTML(display_html))


# testing_log("Test without data")
# testing_log("Test with list", ["a", "b", "c", "d"])
# testing_log(
#     "Test with DataFrame", pd.DataFrame({"col1": [1, 2, 3], "col2": ["x", "y", "z"]})
# )
# testing_log("Test with string", "This is a test string.")

In [0]:
@dataclass
class ResultRecord:
    note_id: int
    search_term_to_note_groups_id: int
    window_start_char_offset: int
    window_end_char_offset: int
    entity_start_offset: int
    entity_end_offset: int
    is_negated: bool


def create_result_record(
    note_id, search_term_to_note_groups_id, ent, doc
) -> ResultRecord:
    result_record = ResultRecord(
        note_id=note_id,
        search_term_to_note_groups_id=search_term_to_note_groups_id,
        window_start_char_offset=_get_window_start_char_offset(ent, doc),
        window_end_char_offset=_get_window_end_char_offset(ent, doc),
        entity_start_offset=ent.start_char,
        entity_end_offset=ent.end_char,
        is_negated=ent._.is_negated,
    )

    return result_record


def _get_window_end_char_offset(ent, doc):
    return doc[min(len(doc), ent.end + 1 + WINDOW_SIZE) - 1].idx


def _get_window_start_char_offset(ent, doc):
    return doc[max(0, ent.start - WINDOW_SIZE)].idx


def _get_local_map_search_term_to_note_group() -> pd.DataFrame:
    extraction_details_csv_path = "/export/home/westerd/_/tedla-hypertension-project-databricks/notebooks/extraction_details.csv"
    local_test_data_df: pd.DataFrame = pd.read_csv(extraction_details_csv_path)

    column_remap = {
        "admission note": "notes_admissions",
        "communication encounter (phone and my health note)": "notes_communication_encounter",
        "discharge summary": "notes_discharge_summary",
        "ecg impression": "notes_ecg_impression",
        "emergency department note": "notes_emergency_department",
        "inpatient note": "notes_inpatient",
        "outpatient note": "notes_outpatient",
        "problem list": "notes_problem_lists",
    }
    # increment the id so local test data matches expected ids
    if "search_term_to_note_groups_id" in local_test_data_df.columns:
        local_test_data_df["search_term_to_note_groups_id"] = (
            local_test_data_df["search_term_to_note_groups_id"].astype(int) + 1
        )
    return local_test_data_df.rename(columns=column_remap)


def get_map_search_term_to_note_group() -> pd.DataFrame:
    """
    Fetch the map_search_term_to_note_group table from Spark and convert to Pandas DataFrame.
    """

    if TESTING:
        return _get_local_map_search_term_to_note_group()
    else:

        return spark.table(
            "workspace_rdtedlak01concordance.default.map_search_term_to_note_group"
        ).toPandas()


class MapSearchTermToNoteGroup:

    JOIN_COLS: List[str] = [
        "notes_admissions",
        "notes_communication_encounter",
        "notes_discharge_summary",
        "notes_ecg_impression",
        "notes_emergency_department",
        "notes_inpatient",
        "notes_outpatient",
        "notes_problem_lists",
    ]

    def __init__(
        self, loader: Callable[[], pd.DataFrame] = get_map_search_term_to_note_group
    ):
        self._search_term_to_note_group: pd.DataFrame = loader()

    @property
    def raw_data(self) -> pd.DataFrame:
        return self._search_term_to_note_group

    @property
    def search_terms(self) -> List[str]:
        return self._search_term_to_note_group["search_term"].tolist()

    def get_search_term_rows(self, row: pd.Series) -> pd.DataFrame:
        row_df: pd.DataFrame = row[MapSearchTermToNoteGroup.JOIN_COLS].to_frame().T
        search_term_rows: pd.DataFrame = self._search_term_to_note_group.merge(
            row_df, on=MapSearchTermToNoteGroup.JOIN_COLS, how="inner"
        )
        search_terms_series = search_term_rows[
            ["search_term_to_note_groups_id", "search_term"]
        ]
        return search_terms_series


def get_note_id_to_note_groups_mapping_source() -> str:
    testing_prefix = ""
    if TESTING:
        testing_prefix = "materialized_"
    return f"default.{testing_prefix}vw_note_id_to_note_groups"


def notes_iterator(batch_size=1000) -> Generator[pd.DataFrame, Any, None]:
    """
    Yields batches of pandas DataFrames of size `batch_size` from the joined Spark DataFrame.
    """

    if TESTING:
        global TEST_DATA_FOR_VALIDATION

        do_populate_test_data = TEST_DATA_FOR_VALIDATION is None

        test_cases_df = []

        for test_data_seed in [
            {"should_be_negated": False, "sent_prefix": "The patient reports "},
            {"should_be_negated": True, "sent_prefix": "The patient does not report "},
        ]:

            sent_prefix = str(test_data_seed["sent_prefix"])
            should_be_negated = bool(test_data_seed["should_be_negated"])

            map_df = _get_local_map_search_term_to_note_group()
            map_df["note_text"] = map_df["search_term"].apply(
                lambda x: f"{sent_prefix}{x} symptoms."
            )
            map_df.insert(0, "note_id", range(1, len(map_df) + 1))

            if do_populate_test_data:

                test_data_df = map_df[
                    [
                        "note_id",
                        "note_text",
                        "search_term_to_note_groups_id",
                        "search_term",
                    ]
                ]
                test_data_df["should_be_negated"] = should_be_negated
                test_data_df["entity_start_offset"] = len(sent_prefix)
                test_data_df["entity_end_offset"] = (
                    test_data_df["entity_start_offset"]
                    + test_data_df["search_term"].str.len()
                )
                test_data_df["windows_start_idx"] = 0
                test_data_df["windows_end_idx"] = (
                    test_data_df["note_text"].str.len() - 1
                )
                test_data_df = test_data_df[
                    [
                        "note_id",
                        "search_term_to_note_groups_id",
                        "entity_start_offset",
                        "entity_end_offset",
                        "windows_start_idx",
                        "windows_end_idx",
                        "should_be_negated",
                    ]
                ]
                if TEST_DATA_FOR_VALIDATION is None:
                    TEST_DATA_FOR_VALIDATION = []
                TEST_DATA_FOR_VALIDATION.extend(test_data_df.to_dict("records"))

            _test_case_df = map_df[
                ["note_id", "note_text", "search_term_to_note_groups_id"]
                + MapSearchTermToNoteGroup.JOIN_COLS
            ]
            test_cases_df.append(_test_case_df)

        yield pd.concat(test_cases_df)
    else:
        # source = get_note_id_to_note_groups_mapping_source()
        notes_skdf: pysqlDataFrame = spark.table(
            "default.materialized_vw_notes_with_timeframe_for_cohort"
        ).select("*")
        total_notes: int = notes_skdf.count()
        for start in range(0, total_notes, batch_size):
            note_batch_skdf: pysqlDataFrame = notes_skdf.limit(
                start + batch_size
            ).subtract(notes_skdf.limit(start))
            local_df: pd.DataFrame = note_batch_skdf.toPandas()
            local_df = local_df[
                [
                    "note_id",
                    "note_text",
                    "notes_problem_lists",
                    "notes_outpatient",
                    "notes_inpatient",
                    "notes_emergency_department",
                    "notes_ecg_impression",
                    "notes_discharge_summary",
                    "notes_communication_encounter",
                    "notes_admissions",
                ]
            ]
            yield local_df


def create_custom_tokenizer(nlp):
    """Create a custom tokenizer that strips punctuation"""
    tokenizer = Tokenizer(nlp.vocab)

    def custom_tokenizer(text):
        # First tokenize normally
        doc = tokenizer(text)

        # Create new tokens with stripped punctuation
        tokens = []
        spaces = []

        for token in doc:
            clean_text = token.text.rstrip(".!?;:,")
            if clean_text:  # Only keep non-empty tokens
                tokens.append(clean_text)
                spaces.append(token.whitespace_)
        return Doc(nlp.vocab, words=tokens, spaces=spaces)

    return custom_tokenizer


class NLPProcessor:
    def __init__(self, map: MapSearchTermToNoteGroup, model="en_core_sci_sm"):
        self._model: str = model
        self._map: MapSearchTermToNoteGroup = map
        nlp = spacy.load(self._model)
        nlp.add_pipe("medspacy_target_matcher")
        target_matcher = nlp.get_pipe("medspacy_target_matcher")
        target_matcher.add(
            [
                TargetRule(search_term, TARGET_ENTITY_LABEL)
                for search_term in map.search_terms
            ]
        )
        nlp.add_pipe("medspacy_context")  # This adds clinical negation
        self._nlp = nlp

    @property
    def pipeline_components(self) -> List[str]:
        return [str(x) for x in self._nlp.pipeline]

    def __call__(self, test_sentence: Iterator[str]) -> Generator[Doc, Any, None]:
        for x in self._nlp.pipe(test_sentence, disable=["ner"]):
            yield x

In [0]:
if TESTING:
    test_notes_iter = notes_iterator(batch_size=1000)
    test_df = next(test_notes_iter)
    testing_log("Test DataFrame", test_df)

In [ ]:
_map = MapSearchTermToNoteGroup(lambda: get_map_search_term_to_note_group())
if TESTING:
    testing_log(
        "First 3 Map Search Term to Note Group DataFrame", _map.raw_data.head(3)
    )
    testing_log("First 3 Search Terms", _map.search_terms[0:3])
nlp_processor = NLPProcessor(_map)

In [0]:
from typing import List
from spacy.tokens.span import Span

testing_log("NLP Pipeline Components", nlp_processor.pipeline_components)


def create_result_records(doc: Doc, row: pd.Series) -> List[ResultRecord]:
    search_terms_series: pd.DataFrame = _map.get_search_term_rows(row)
    symptoms_of_interest: list[Span] = [
        ent for ent in doc.ents if ent.label_ == TARGET_ENTITY_LABEL
    ]
    records: List[ResultRecord] = []
    for ent in symptoms_of_interest:
        search_term_mask = search_terms_series["search_term"] == ent.text
        search_term_to_note_groups_id = int(
            search_terms_series[search_term_mask]["search_term_to_note_groups_id"].iloc[
                0
            ]
        )
        result_record: ResultRecord = create_result_record(
            row["note_id"], search_term_to_note_groups_id, ent, doc
        )
        records.append(result_record)
    return records

In [0]:
from abc import ABC, abstractmethod
from dataclasses import asdict


class WriterBase(ABC):

    def __init__(self) -> None:
        pass

    @abstractmethod
    def __call__(self, records: List[ResultRecord]):
        raise NotImplementedError()


class MockWriter(WriterBase):

    def __init__(self) -> None:
        pass

    def __call__(self, records: List[ResultRecord]):
        for record in records:
            note_id = record.note_id
            test_data_collection = [
                x
                for x in TEST_DATA_FOR_VALIDATION
                if x["note_id"] == note_id
                and x["should_be_negated"] is record.is_negated
            ]
            assert len(test_data_collection) == 1
            test_data = test_data_collection[0]

            if (
                int(record.search_term_to_note_groups_id)
                != test_data["search_term_to_note_groups_id"]
            ):
                raise ValueError("search_term_to_note_groups_id")
            if record.entity_start_offset != test_data["entity_start_offset"]:
                raise ValueError("entity_start_offset")
            if record.entity_end_offset != test_data["entity_end_offset"]:
                raise ValueError("entity_end_offset")
            if record.window_start_char_offset != test_data["windows_start_idx"]:
                raise ValueError("window_start_char_offset")
            if record.window_end_char_offset != test_data["windows_end_idx"]:
                raise ValueError("window_end_char_offset")
            if record.is_negated != test_data["should_be_negated"]:
                raise ValueError("is_negated")

            display(f"{record.note_id}: {record.is_negated}")


class ResultWriter(WriterBase):

    def __init__(self) -> None:
        pass

    def __call__(self, records: List[ResultRecord]):
        if not records:
            return

        # Convert ResultRecord objects to list of dicts
        rows = []
        if not all(isinstance(x, ResultRecord) for x in records):
            raise ValueError("Unexpected entry in records, not type ResultRecord")
        
        # Create Spark DataFrame and append to table
        sdf = spark.createDataFrame([asdict(r) for r in records])
        table_name = "default.results"
        db, tbl = table_name.split(".", 1)
        # If table doesn't exist, create it with the same schema (empty) first
        if spark.sql(f"SHOW TABLES IN {db} LIKE '{tbl}'").count() == 0:
            sdf.limit(0).write.saveAsTable(table_name)
        # Append the new rows
        sdf.write.mode("append").saveAsTable(table_name)

In [ ]:
class NoteBatchProcessor:

    def __init__(self, writer: WriterBase) -> None:
        self._writer: WriterBase = writer

    def __call__(self, batch_df: pd.DataFrame):

        all_records: List[ResultRecord] = []

        def iterate_notes() -> Generator[str, Any, None]:
            for note_idx in range(len(batch_df)):
                yield batch_df.iloc[note_idx]["note_text"]

        doc: Doc
        for note_idx, doc in enumerate(nlp_processor(iterate_notes())):
            row: pd.Series = batch_df.iloc[note_idx]
            records: List[ResultRecord] = create_result_records(doc, row)
            all_records.extend(records)

        self._writer(all_records)

In [ ]:
note_batch_processor = NoteBatchProcessor(MockWriter())

for notes_df in notes_iterator(1000):
    note_batch_processor(notes_df)